# 第5週 AI 分類入門與成果發表

**學員版：先做示範，再完成練習。**

每週180分鐘，含10分鐘休息。每個W標號對應PPT與Word同一單元。

教材對照：CH11PPT 11-1至11-3；CH15PPT 15-2概念；CH16PPT 16-1流程。色彩最近中心分類器為本課新增簡化實作

學習目標：
- 能區分規則、訓練與推論
- 能分開訓練資料及保留測試資料
- 能完成分類、評估並解釋限制

先按Shift+Enter逐格執行；交件前重新啟動核心並全部執行。練習格預設以註解保存，請取消註解後完成。請勿將未完成的函數覆蓋示範函數後繼續執行。

## 課前套件
由講師先依總覽安裝套件。核心程式不下載模型或開啟攝影機；色卡由程式自行產生。輸出檔案寫在目前工作目錄的outputs資料夾。

In [ ]:
import sys
print("Python", sys.version.split()[0])

## W5-01 AI 概念與本週成果

000–010 分鐘，10分鐘。

- AI 是廣泛領域
- 機器學習從資料估計參數
- 深度學習是機器學習的一支

**操作任務**：說明手寫閾值與從樣本估計平均中心的差別。

**完成標準**：前者參數由人設定，後者中心由帶標籤訓練資料計算。

## W5-02 特徵與標籤

010–025 分鐘，15分鐘。

- 特徵描述輸入資料
- 標籤是已知類別
- 今日特徵為平均R、G、B

**操作任務**：執行資料準備，印出第一張色卡與特徵，分辨影像、特徵、標籤。

In [ ]:
import numpy as np
from PIL import Image
from IPython.display import display
from pathlib import Path
import csv

def make_card(rgb_values):
    image = np.zeros((24, 24, 3), dtype=np.uint8)
    image[:] = rgb_values
    return image

def features(image_rgb):
    return image_rgb.astype(float).mean(axis=(0, 1))

card = make_card([220, 30, 30])
display(Image.fromarray(card))
print(card.shape, features(card))

### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [ ]:
# # 建立一張綠色色卡，印出其特徵
# my_card = make_card([30, 220, 30])
# print(features(my_card))

**完成標準**：影像(24,24,3)；特徵(3,)；標籤為red等類別文字。

## W5-03 訓練集與保留測試集

025–045 分鐘，20分鐘。

- 訓練集用來計算中心
- 測試集保留到模型固定後
- 同張圖片重複放兩邊會洩漏答案

**操作任務**：數出各類訓練與測試筆數，確認沒有相同色卡同時出現在兩側。

In [ ]:
train_values = [[220,30,30], [180,20,20], [240,50,50],
                [30,220,30], [20,180,20], [50,240,50],
                [30,30,220], [20,20,180], [50,50,240]]
y_train = np.array(["red"] * 3 + ["green"] * 3 + ["blue"] * 3)
test_values = [[200,25,25], [230,40,40], [25,200,25],
               [40,230,40], [25,25,200], [40,40,230]]
y_test = np.array(["red", "red", "green", "green", "blue", "blue"])
X_train = np.array([features(make_card(v)) for v in train_values])
X_test = np.array([features(make_card(v)) for v in test_values])
print(X_train.shape, X_test.shape)

### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [ ]:
# # 比較訓練與測試是否含完全相同的RGB數值
# print(set(map(tuple, train_values)) & set(map(tuple, test_values)))

**完成標準**：每類訓練3筆、測試2筆，共9與6筆。

## W5-04 訓練最近中心分類器

045–060 分鐘，15分鐘。

- 同類特徵取平均
- 中心是從訓練資料算出的參數
- 三個類別各有一個中心

**操作任務**：計算三個中心，手算紅色中心第一個通道與程式核對。

In [ ]:
classes = np.unique(y_train)
centroids = []
for label in classes:
    class_rows = X_train[y_train == label]
    centroids.append(class_rows.mean(axis=0))
centroids = np.array(centroids)
for label, center in zip(classes, centroids):
    print(label, np.round(center, 2))

### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [ ]:
# # 手算紅色中心R通道
# manual_red_r = (220 + 180 + 240) / 3
# print(manual_red_r)

**完成標準**：紅色中心約[213.33,33.33,33.33]，共3個中心。

## W5-05 休息與模型保存

060–070 分鐘，10分鐘。

- 休息10分鐘
- 不要用測試資料重算中心
- 保留資料分組

**操作任務**：存檔與休息。

**完成標準**：中心只由X_train與y_train計算。

## W5-06 推論與距離

070–090 分鐘，20分鐘。

- 新特徵與各中心比較
- 平方差總和最小者勝出
- 距離不是機率

**操作任務**：預測[200,25,25]，查看三個距離與最小值索引。

In [ ]:
def predict_one(feature):
    distances = []
    for center in centroids:
        distances.append(float(np.sum((feature - center) ** 2)))
    index = int(np.argmin(distances))
    return str(classes[index]), distances

prediction, distances = predict_one(X_test[0])
print(prediction, np.round(distances, 2))

### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [ ]:
# # 預測一張藍色卡
# print(predict_one(features(make_card([20, 20, 210])))[0])

**完成標準**：預測red；最小距離所對應的類別是red。

## W5-07 測試與正確率

090–110 分鐘，20分鐘。

- 每筆預測對照已知標籤
- 正確率是答對筆數除以總筆數
- 小而簡單的測試無法代表所有情況

**操作任務**：列出六筆結果，計算correct、total與accuracy。

In [ ]:
predictions = []
for feature in X_test:
    label, _ = predict_one(feature)
    predictions.append(label)
predictions = np.array(predictions)
correct = int(np.sum(predictions == y_test))
accuracy = correct / len(y_test)
for truth, pred in zip(y_test, predictions):
    print("真實", truth, "預測", pred)
print(correct, len(y_test), accuracy)

### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [ ]:
# # 人工驗算：若六筆中答對四筆，正確率多少？
# example_accuracy = 4 / 6
# print(example_accuracy)

**完成標準**：六筆預期皆正確；6/6=1.0，此結果僅適用這組色卡。

## W5-08 陌生樣本與失敗案例

110–125 分鐘，15分鐘。

- 模型只認得既定三類
- 灰色也會被迫選一類
- 同距離時argmin選第一個

**操作任務**：預測[128,128,128]並觀察距離；記錄它應屬未學過的灰色。

In [ ]:
unknown = make_card([128, 128, 128])
display(Image.fromarray(unknown))
unknown_label, unknown_distances = predict_one(features(unknown))
print("灰色被分為", unknown_label)
print(np.round(unknown_distances, 2))

### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [ ]:
# # 再測一張黃色卡，記錄限制
# yellow_label, yellow_distances = predict_one(features(make_card([230, 230, 20])))
# print(yellow_label, np.round(yellow_distances, 2))

**完成標準**：距離相同，classes排序第一個為blue；這是平手規則，不是灰色理解。

## W5-09 教材 AI 應用對照

125–140 分鐘，15分鐘。

- 人臉偵測找位置，識別判身分
- 手勢可先估關鍵點再套規則
- 預訓練模型仍需要檢查結果

**操作任務**：將「找出臉框」「認出小安」「紅色卡分類」配對成偵測、識別、分類。

**完成標準**：找臉框是偵測，認出小安是身分識別，紅色卡是分類。

## W5-10 期末專題與成果發表

140–160 分鐘，20分鐘。

- 保存每筆真實標籤與預測
- 報告6筆保留測試結果
- 展示至少1個失敗或陌生案例

**操作任務**：保存predictions.csv及模型中心；完成3句結論，兩組互講或每組90秒發表。

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
print("請整理本週成果")

### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [ ]:
# # 保存六筆預測CSV與模型中心
# conclusion = "請填寫測試筆數、表現與限制"
# print(conclusion)

**完成標準**：CSV有6筆結果；中心以npz保存；結論包含6/6、人工色卡限制、灰色被迫分類。

## W5-11 專題評量與重跑

160–170 分鐘，10分鐘。

- 程式可從乾淨核心重跑
- 資料與方法說明完整
- 限制說明與分數同樣重要

**操作任務**：依總覽評量表互評，提交Notebook、CSV與結論。

**完成標準**：可重跑且輸出完整；結論沒有將距離稱為機率，也沒有宣稱通用物體辨識。

## W5-12 五週回顧與下一步

170–180 分鐘，10分鐘。

- 變數、控制、函數、資料、影像
- 模型品質需要資料與評估
- 延伸閱讀教材12至16章

**操作任務**：離堂回答：訓練與推論差別、為何分開測試、灰色案例的限制。

**完成標準**：訓練估計中心，推論使用固定中心；保留測試檢查未用來訓練的資料；模型不能自動拒絕未知類別。